In [ ]:
#Split didakta-english.md by § dividers into separate files
import re
from pathlib import Path

INPUT_FILE = Path("didakta-english.md")
OUTPUT_DIR = Path("didakta-english-split-entries")

content = INPUT_FILE.read_text(encoding='utf-8')

# Find all § entries
# Pattern: ### **§CODE. Name**
pattern = r'### \*\*§([A-Za-z]+)\. ([^\*]+)\*\*.*?\n\n((?:(?!### \*\*§).)*?)(?=### \*\*§|$)'

entries = []
for match in re.finditer(pattern, content, re.DOTALL):
    code = match.group(1)
    name = match.group(2).strip()
    body = match.group(3).strip()
    
    entries.append({
        'code': code,
        'name': name,
        'body': body
    })

print(f"Found {len(entries)} entries")

# Create output directory
OUTPUT_DIR.mkdir(exist_ok=True)

# Write each entry to its own file
for entry in entries:
    code = entry['code']
    name = entry['name']
    body = entry['body']
    
    filename = f"{code}.md"
    filepath = OUTPUT_DIR / filename
    
    # Create markdown content
    md_content = f"# §{code}. {name}\n\n{body}\n"
    
    filepath.write_text(md_content, encoding='utf-8')

print(f" Created {len(entries)} files")


Found 125 entries
 Created 125 files


In [ ]:
#Removing hyperlinks from didakta-english.md
INPUT_FILE = Path("didakta-english.md")
OUTPUT_FILE = Path("didakta-english-no-links.md")


def strip_markdown_inline_links(text: str) -> str:
    out = []
    i = 0
    n = len(text)

    while i < n:
        if text[i] != '[':
            out.append(text[i])
            i += 1
            continue

        # Parse [label] with nested brackets support.
        j = i + 1
        bdepth = 1
        while j < n and bdepth > 0:
            if text[j] == '\\':
                j += 2
                continue
            if text[j] == '[':
                bdepth += 1
            elif text[j] == ']':
                bdepth -= 1
            j += 1

        if bdepth != 0:
            out.append(text[i])
            i += 1
            continue

        label = text[i + 1:j - 1]

        # Must be immediately followed by (url)
        if j >= n or text[j] != '(':
            out.append(text[i])
            i += 1
            continue

        k = j + 1
        pdepth = 1
        while k < n and pdepth > 0:
            if text[k] == '\\':
                k += 2
                continue
            if text[k] == '(':
                pdepth += 1
            elif text[k] == ')':
                pdepth -= 1
            k += 1

        if pdepth != 0:
            out.append(text[i])
            i += 1
            continue

        # Valid link found: keep label, drop URL.
        out.append(label)
        i = k

    return ''.join(out)


content = INPUT_FILE.read_text(encoding='utf-8')
content = strip_markdown_inline_links(content)
OUTPUT_FILE.write_text(content, encoding='utf-8')


143346

In [ ]:
# Create a structured no-links version (single consolidated cell)

INPUT_FILE = Path("didakta-english-no-links.md")
OUTPUT_FILE = Path("didakta-english-no-links-structured.md")
PRESERVE_FOOTNOTES = True

text = INPUT_FILE.read_text(encoding="utf-8")

# Start from first real grammar heading to drop the TOC block
content_start_match = re.search(r"(?m)^# \*\*.+?\*\*(?:\s+\{#.*?\})?\s*$", text)
if not content_start_match:
    raise ValueError("Could not find grammar content start.")

content_start = content_start_match.start()
toc_start_match = re.search(r"(?m)^Nominative\s*$", text[:content_start])
if toc_start_match:
    intro = text[:toc_start_match.start()].strip()
else:
    intro = text[:content_start].strip()

body = text[content_start:]

# Remove anchor fragments; optionally remove footnotes
body = re.sub(r"\s*\{#.*?\}", "", body)
if not PRESERVE_FOOTNOTES:
    body = re.sub(r"\[\^\d+\]", "", body)

# Normalize headings
body = re.sub(r"(?m)^# \*\*(.+?)\*\*\s*$", r"## \1", body)
body = re.sub(r"(?m)^### \*\*(.+?)\*\*\s*$", r"### \1", body)

# Remove orphan/empty headings and tidy spacing
body = re.sub(r"(?m)^###\s*$", "", body)
body = re.sub(r"(?m)^#\s*$", "", body)
body = re.sub(r"\n{3,}", "\n\n", body).strip()

subtitle = (
    "This version removes the table of contents and keeps the main grammar content with footnotes preserved."
    if PRESERVE_FOOTNOTES
    else "This version removes the table of contents and keeps only the main grammar content in a cleaner reading order."
)

new_text = (
    "# Didakta Grammar for Annotation (English, No Hyperlinks, Structured)\n\n"
    + subtitle
    + "\n\n"
    + intro
    + "\n\n---\n\n"
    + body
    + "\n"
)

OUTPUT_FILE.write_text(new_text, encoding="utf-8")


✅ Rebuilt: didakta-english-no-links-structured.md
Footnotes preserved: True


In [ ]:
# Do the same pipeline for didakta-kurdish.md and didakta-persian.md

PRESERVE_FOOTNOTES = True


def strip_markdown_inline_links(text: str) -> str:
    """Convert [label](url) to label using a parser (handles nested () in URLs)."""
    out = []
    i = 0
    n = len(text)

    while i < n:
        if text[i] != '[':
            out.append(text[i])
            i += 1
            continue

        j = i + 1
        bdepth = 1
        while j < n and bdepth > 0:
            if text[j] == '\\':
                j += 2
                continue
            if text[j] == '[':
                bdepth += 1
            elif text[j] == ']':
                bdepth -= 1
            j += 1

        if bdepth != 0:
            out.append(text[i])
            i += 1
            continue

        label = text[i + 1:j - 1]

        if j >= n or text[j] != '(':
            out.append(text[i])
            i += 1
            continue

        k = j + 1
        pdepth = 1
        while k < n and pdepth > 0:
            if text[k] == '\\':
                k += 2
                continue
            if text[k] == '(':
                pdepth += 1
            elif text[k] == ')':
                pdepth -= 1
            k += 1

        if pdepth != 0:
            out.append(text[i])
            i += 1
            continue

        out.append(label)
        i = k

    return ''.join(out)


def make_structured_text(text: str, preserve_footnotes: bool = True) -> str:
    """Remove TOC and normalize heading structure."""
    content_start_match = re.search(r"(?m)^# \*\*.+?\*\*(?:\s+\{#.*?\})?\s*$", text)
    if not content_start_match:
        raise ValueError("Could not find grammar content start.")

    content_start = content_start_match.start()
    toc_start_match = re.search(r"(?m)^Nominative\s*$", text[:content_start])
    if toc_start_match:
        intro = text[:toc_start_match.start()].strip()
    else:
        intro = text[:content_start].strip()

    body = text[content_start:]
    body = re.sub(r"\s*\{#.*?\}", "", body)

    if not preserve_footnotes:
        body = re.sub(r"\[\^\d+\]", "", body)

    body = re.sub(r"(?m)^# \*\*(.+?)\*\*\s*$", r"## \1", body)
    body = re.sub(r"(?m)^### \*\*(.+?)\*\*\s*$", r"### \1", body)
    body = re.sub(r"(?m)^###\s*$", "", body)
    body = re.sub(r"(?m)^#\s*$", "", body)
    body = re.sub(r"\n{3,}", "\n\n", body).strip()

    subtitle = (
        "This version removes the table of contents and keeps the main grammar content with footnotes preserved."
        if preserve_footnotes
        else "This version removes the table of contents and keeps only the main grammar content in a cleaner reading order."
    )

    return (
        "# Didakta Grammar for Annotation (No Hyperlinks, Structured)\n\n"
        + subtitle
        + "\n\n"
        + intro
        + "\n\n---\n\n"
        + body
        + "\n"
    )


for lang in ["kurdish", "persian"]:
    input_file = Path(f"didakta-{lang}.md")
    no_links_file = Path(f"didakta-{lang}-no-links.md")
    structured_file = Path(f"didakta-{lang}-no-links-structured.md")

    if not input_file.exists():
        print(f"⚠ Missing input: {input_file}")
        continue

    original = input_file.read_text(encoding="utf-8")
    no_links = strip_markdown_inline_links(original)
    no_links_file.write_text(no_links, encoding="utf-8")

    structured = make_structured_text(no_links, preserve_footnotes=PRESERVE_FOOTNOTES)
    structured_file.write_text(structured, encoding="utf-8")



✅ kurdish: didakta-kurdish-no-links.md + didakta-kurdish-no-links-structured.md
✅ persian: didakta-persian-no-links.md + didakta-persian-no-links-structured.md


In [ ]:
# Split didakta-persian.md and didakta-kurdish.md into per-entry files
entry_pattern = re.compile(
    r"^#{2,3}\s+\*\*§([A-Za-z]+)\.\s*(.*?)\*\*.*?\n\n(.*?)(?=^#{2,3}\s+\*\*§[A-Za-z]+\.|\Z)",
    re.MULTILINE | re.DOTALL,
)

for lang in ["persian", "kurdish"]:
    input_file = Path(f"didakta-{lang}.md")
    output_dir = Path(f"didakta-{lang}-split-entries")

    if not input_file.exists():
        print(f"⚠ Missing input: {input_file}")
        continue

    content = input_file.read_text(encoding="utf-8")
    matches = list(entry_pattern.finditer(content))

    entries = []
    for match in matches:
        code = match.group(1).strip()
        name = match.group(2)
        body = match.group(3).strip()

        # Clean display name by removing footnote markers in headings.
        name = re.sub(r"\[\^\d+\]", "", name)
        name = re.sub(r"\s+", " ", name).strip()

        entries.append({"code": code, "name": name, "body": body})

    output_dir.mkdir(exist_ok=True)

    for entry in entries:
        filepath = output_dir / f"{entry['code']}.md"
        md_content = f"# §{entry['code']}. {entry['name']}\n\n{entry['body']}\n"
        filepath.write_text(md_content, encoding="utf-8")


✅ persian: 127 files in didakta-persian-split-entries
✅ kurdish: 127 files in didakta-kurdish-split-entries
